In [1]:
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import numpy as np

In [2]:
CPS_PATH = '../data/cps_00018.csv'
EXPOSURE_SCORES_PATH = '../data/final_exposure.csv'

In [16]:
# CPS occupation
OCC = 'OCC'
# Employment Status
EMPSTAT = 'EMPSTAT'
# CPS age
AGE = 'AGE'
# Young/Entry Workers Column
Y = '18_25'
# CPS weight  
WEIGHT = 'WTFINL'                  
# Scores occupation       
INDEX_OCC = 'CPS Code'
# All score columns
INDEXS = ['alpha', 'beta', 'gamma']

In [17]:
df_raw = pd.read_csv(CPS_PATH)
exposure_scores = pd.read_csv(EXPOSURE_SCORES_PATH)

# Filter out OCC 0, NA, and unemployed
df_raw = df_raw[
    (df_raw[OCC] > 0) & 
    (df_raw[OCC].notna()) & 
    (df_raw[EMPSTAT].isin([10, 12]))
]

In [14]:
# Convert both OCC and CPS Code to string columns
df_raw[OCC] = df_raw[OCC].astype(float).astype(int).astype(str)
exposure_scores[INDEX_OCC] = exposure_scores[INDEX_OCC].astype(float).astype(int).astype(str)

In [20]:
# Merge exposure scores onto the dataset
df = df_raw.merge(exposure_scores,
                     left_on=OCC,
                     right_on=INDEX_OCC,
                     how='left',
                     indicator=True)

# 32203 Rows do not merge
failed = df['_merge'] == 'left_only'
failed_to_merge = df[failed]

# 13 occupation codes did not merge
# They are residuals (All Other) or Military occupations (9840)
# Onet does not track them so they are being dropped
# https://www.onetcenter.org/dl_files/Taxonomy2019_Summary.pdf
# See citation 4 on page 10

df = df.drop(columns=[INDEX_OCC, "CPS Title", "_merge"])

df_scores = df[~failed]

In [21]:
# Construct panels at 4 different window lengths, check for cell size filters.
# Since the actual proportions per occupation are fairly low,
# Many low N and low n cells provide unreliable point estimates of the true proportion

# Create count of young workers
df_scores[Y] = df_scores[AGE].between(18, 25).copy()

# Aggregate to quarter, half year and full year levels using treatment date as time = 0
treat_year = 2022
treat_month = 11

df_scores['MONTHS_FROM_TREAT'] = (df_scores["YEAR"] - treat_year) * 12 + (df_scores['MONTH'] - treat_month)
df_scores['QUARTER'] = df_scores['MONTHS_FROM_TREAT'] // 3
df_scores['HALF_YEAR'] = df_scores['MONTHS_FROM_TREAT'] // 6
df_scores['YEARS_FROM_TREAT'] = df_scores['MONTHS_FROM_TREAT'] // 12

# Drop trailing months when aggregating:

# Take number of months within every period specification and trim if the number of months is less than the specified window_length
# Tolerance -- we keep 2025H2 and 2025 but not 2025Q4 because of the October CPS
def drop_thin_periods(df, period_col, window_length, tolerance=1):
    counts = df.groupby(period_col)[['YEAR','MONTH']].apply(lambda g: g.drop_duplicates().shape[0])
    keep = counts[counts >= window_length - tolerance].index
    return df[df[period_col].isin(keep)]

df_q = drop_thin_periods(df_scores, 'QUARTER', 3, tolerance=0)
df_h = drop_thin_periods(df_scores, 'HALF_YEAR', 6, tolerance=1)
df_y = drop_thin_periods(df_scores, 'YEARS_FROM_TREAT', 12, tolerance=1)

In [22]:
# Function to get occupations that pass the filter in each period in each window size
def compute_cell_filters(df_w, window_col, n_min=30, np_min=5):
    #We only filter on pre-treatment data
    df_w = df_w[(df_w[window_col] < 0)]

    cells_w = df_w.groupby(['OCC', window_col]).agg(
        N=(AGE, 'size'),
        np=(Y, 'sum'))
    cells_w['nq'] = cells_w['N'] - cells_w['np']

    cell_counts_N = cells_w['N'].unstack(fill_value=0)
    cell_counts_np = cells_w['np'].unstack(fill_value=0)
    cell_counts_nq = cells_w['nq'].unstack(fill_value=0)

    valid = (cell_counts_N >= n_min) & (cell_counts_np >= np_min) & (cell_counts_nq >= np_min)
    passing_occ = valid.index[valid.all(axis=1)]
    pass_counts = valid.sum(axis=0)

    return {
        "passing_occ": passing_occ,            # List of OCCs that pass in ALL windows
        "pass_counts": pass_counts,            # Series: total passing OCCs per window
        "valid_matrix": valid,                 # Boolean DataFrame (OCC x Window)
        "cell_counts": {                       # Raw count matrices
            "N": cell_counts_N,
            "np": cell_counts_np,
            "nq": cell_counts_nq
        }
    }   

In [23]:
windows = {
    'monthly': (df_scores, 'MONTHS_FROM_TREAT'),
    'quarterly': (df_q, 'QUARTER'),
    'half-year': (df_h, 'HALF_YEAR'),
    'full-year': (df_y, 'YEARS_FROM_TREAT'),
}

filter_results = {}

for name, (df_w, window_col) in windows.items():
    filter_results[name] = compute_cell_filters(df_w, window_col)
    n_passing = len(filter_results[name]['passing_occ'])
    print(f"PASSING occupations ({name}): {n_passing}, Percent: {(n_passing/513)*100}%")
    

PASSING occupations (monthly): 120, Percent: 23.391812865497073%
PASSING occupations (quarterly): 244, Percent: 47.563352826510716%
PASSING occupations (half-year): 349, Percent: 68.03118908382066%
PASSING occupations (full-year): 462, Percent: 90.05847953216374%


In [24]:
# Testing for selection bias
# Do cell-filter failing and passing occupations differ in their exposure scores

for index in INDEXS:
    print(f'FOR INDEX {index}:')
    results_exposure = []
    for name, (df_w, window_col) in windows.items():
        df_w = df_w[(df_w[window_col] < 0)]
        passing_w = filter_results[name]['passing_occ']
        t_stat, p_val = stats.ttest_ind(
            df_w[df_w['OCC'].isin(passing_w)].groupby('OCC')[index].mean(),
            df_w[~df_w['OCC'].isin(passing_w)].groupby('OCC')[index].mean(),
            equal_var=False
        )
        results_exposure.append({'window': name, 't_stat': t_stat, 'p_val': p_val})
    results_exposure_df = pd.DataFrame(results_exposure)
    print(results_exposure_df)


# No statistically different results between filtered and unfiltered groups at any level across any index

FOR INDEX alpha:
      window    t_stat     p_val
0    monthly -0.666449  0.505815
1  quarterly  0.601712  0.547635
2  half-year  1.560753  0.119464
3  full-year  1.176186  0.244079
FOR INDEX beta:
      window    t_stat     p_val
0    monthly -0.442406  0.658665
1  quarterly  1.138325  0.255525
2  half-year  1.360758  0.174496
3  full-year  0.373462  0.710035
FOR INDEX gamma:
      window    t_stat     p_val
0    monthly -0.256198  0.798066
1  quarterly  1.183407  0.237205
2  half-year  1.005306  0.315486
3  full-year -0.149507  0.881627


In [25]:
results_share = []
for name, (df_w, window_col) in windows.items():
    df_w = df_w[(df_w[window_col] < 0)]
    passing_w = filter_results[name]['passing_occ']
    
    passing_vals = df_w[df_w['OCC'].isin(passing_w)].groupby('OCC')[Y].mean()
    failing_vals = df_w[~df_w['OCC'].isin(passing_w)].groupby('OCC')[Y].mean()
    t_stat, p_val = stats.ttest_ind(passing_vals, failing_vals, equal_var=False)
    diff = passing_vals.mean() - failing_vals.mean()
    
    # Baseline: average of occupation-level means across all occupations
    overall_occ_mean = df_w.groupby('OCC')[Y].mean().mean()
    pct_diff = (diff / overall_occ_mean) * 100
    results_share.append({
        'window': name, 
        't_stat': t_stat, 
        'p_val': p_val, 
        'diff': diff, 
        'pct_diff': pct_diff
    })

print(pd.DataFrame(results_share))

# In every window filtered out cells have less young workers
# However based on CLT for bionmial proportions these estimates may not be accurate
# Will include high N groups in robustness check even if they are low np

      window     t_stat         p_val      diff   pct_diff
0    monthly   5.177717  5.981009e-07  0.049063  44.319511
1  quarterly   6.592006  1.148271e-10  0.049397  44.621004
2  half-year   7.683247  1.332425e-13  0.056761  51.272874
3  full-year  10.122608  3.707722e-16  0.085344  77.092529


In [26]:
for name, (df_w, window_col) in windows.items():
        df_w = df_w[(df_w[window_col] < 0)]
        emp_share = df_w.groupby('OCC')[WEIGHT].sum()
        passing_w = filter_results[name]['passing_occ']
        print(f"Passing occ share of total employment of sample({name}: {(emp_share[passing_w].sum() / emp_share.sum())*100}%")

Passing occ share of total employment of sample(monthly: 71.65888404131977%
Passing occ share of total employment of sample(quarterly: 88.59443990004257%
Passing occ share of total employment of sample(half-year: 94.98647816468574%
Passing occ share of total employment of sample(full-year: 98.88645453056789%


In [27]:
passing_occs_df = pd.concat([
    pd.DataFrame({'window': window_name, 'OCC': list(res['passing_occ'])})
    for window_name, res in filter_results.items()
], ignore_index=True)

passing_occs_df.to_json('../data/checkpoints/passing_occs.json', orient='records', indent=2)

In [28]:
# Create 3 panels needed for final

final_windows = {
    'quarterly': df_q,
    'half-year': df_h
}
panels = {}

for window, df_w in final_windows.items():
    window_col = 'HALF_YEAR'
    if window == 'quarterly': window_col = 'QUARTER'
    panel_w = df_w.groupby(['OCC', window_col]).agg(
        N=(AGE, 'size'),
        np=(Y, 'sum'),
        **{col: (col, 'first') for col in INDEXS},
    ).reset_index()

    panel_w['Y_SHARE'] = panel_w['np'] / panel_w['N']
    panel_w['arcsinh_np'] = np.arcsinh(panel_w['np'])
    panel_w['arcsinh_N'] = np.arcsinh(panel_w['N'])

    for col in INDEXS:
        quintile_col = f"{col}_quintile"
        panel_w[quintile_col] = pd.qcut(panel_w[col], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
        dummies = pd.get_dummies(panel_w[quintile_col], prefix=quintile_col, dtype=int)
        panel_w[dummies.columns] = dummies

    panels[window] = panel_w



In [29]:
for col in INDEXS:
    quintile_col = f"{col}_quintile"

    stats_quintiles = panels['half-year'].groupby(quintile_col).agg(
        min=(col, 'min'),
        max=(col, 'max'),
        mean=(col, 'mean'),
        median=(col, 'median'),
        stdev=(col, 'std'),
        total_obs=(col, 'count'),
        unique_occ_count=(OCC, 'nunique')
    )
    stats_quintiles.index.name = 'Quintile'
    
    print(f"\n--- Statistics for column: '{col}' ---")
    print(stats_quintiles.to_string())


--- Statistics for column: 'alpha' ---
               min       max      mean    median     stdev  total_obs  unique_occ_count
Quintile                                                                               
1         0.000000  0.033333  0.006620  0.000000  0.011170        926               103
2         0.035714  0.078947  0.058162  0.059121  0.012881        936               104
3         0.080000  0.133333  0.102578  0.100000  0.014765        909               101
4         0.135543  0.213845  0.170002  0.169274  0.022363        927               103
5         0.214286  0.950000  0.417054  0.340580  0.206619        918               102

--- Statistics for column: 'beta' ---
               min       max      mean    median     stdev  total_obs  unique_occ_count
Quintile                                                                               
1         0.000000  0.105263  0.052763  0.055717  0.032310        936               104
2         0.105769  0.261364  0.179608  0

In [30]:
panel_q = panels['quarterly'][panels['quarterly']['OCC'].isin(filter_results['half-year']['passing_occ'])]
panel_h = panels['half-year'][panels['half-year']['OCC'].isin(filter_results['half-year']['passing_occ'])]

In [31]:
for col in INDEXS:
    quintile_col = f"{col}_quintile"

    stats = panel_q.groupby(quintile_col).agg(
        min=(col, 'min'),
        max=(col, 'max'),
        mean=(col, 'mean'),
        median=(col, 'median'),
        stdev=(col, 'std'),
        total_obs=(col, 'count'),
        unique_occ_count=(OCC, 'nunique')
    )
    stats.index.name = 'Quintile'
    
    print(f"\n--- Statistics for column: '{col}' ---")
    print(stats.to_string())


--- Statistics for column: 'alpha' ---
               min       max      mean    median     stdev  total_obs  unique_occ_count
Quintile                                                                               
1         0.000000  0.033333  0.006596  0.000000  0.011059       1173                69
2         0.035714  0.078947  0.058416  0.059418  0.012737       1207                71
3         0.080357  0.133333  0.102875  0.103371  0.013782       1020                60
4         0.135543  0.213845  0.170959  0.170732  0.022577       1241                73
5         0.214286  0.900000  0.419941  0.333333  0.204848       1292                76

--- Statistics for column: 'beta' ---
               min       max      mean    median     stdev  total_obs  unique_occ_count
Quintile                                                                               
1         0.000000  0.105263  0.049885  0.050000  0.032764       1190                70
2         0.107143  0.261364  0.183738  0

In [32]:
panel_h.to_csv('../data/checkpoints/ready/panel_h.csv', index=False)
panel_q.to_csv('../data/checkpoints/ready/panel_q.csv', index=False)
panels['half-year'].to_csv('../data/checkpoints/ready/panel_full.csv', index=False)